# HostWise - Track C: Data Quality (five cities)

**Cities:** Barcelona, London, Madrid, Paris, Rome (New York excluded).

**Goal:** measure every data-quality issue that affects the Day 2 preprocessing - missing values, target missingness, currency, leakage, duplicates, invalid values, stay length, structural missingness and price outliers. This notebook is **diagnostic only**: it does not modify, clean or save data.

**Notebook guide**

| Section | Question |
|---|---|
| C1 | How much data is missing, and how does that change by population? |
| C2 | Why is the target missing - inactive listings or random gaps? |
| C3 | Which currency is each city's price in? |
| C4 | Which columns leak the target? |
| C5 | Are there duplicate listings, and how concentrated is supply among hosts? |
| C6 | Which values are impossible, implausible or contradictory? |
| C7 | Are some listings mid- or long-term rentals rather than short stays? |
| C8 | Is missingness structural (meaningful) or random? |
| C9 | How extreme are the price tails, and which outlier rule fits each city? |

All price statistics are reported **per city in local currency**. Prices from different currencies are never pooled or compared directly.

## 0. Setup

The setup cells are identical in every HostWise notebook, so all tracks use the same data and the same population definitions.

| Name | Definition |
|---|---|
| `df_all` | All listings in `hostwise_5cities.csv` (five cities) |
| `df_active` | `source == "city scrape"`: listings present in the current scrape |
| `df_priced` | `df_active` with a parseable price |
| `df_scope` | `df_priced` with `minimum_nights <= 29` (short-term scope, pending team approval) |

Conventions: the parsed price is `price_num` and the log target is `log_price` (`log1p`, inverse `expm1`). Prices are in local currency (London GBP, other cities EUR).

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid")

In [5]:
DATA_CANDIDATES = [
    Path("hostwise_combined.csv"),
]

FIG_DIR = Path("../reports/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

CITY_ORDER = ["london", "paris", "rome", "madrid", "barcelona"]
CURRENCY = {"london": "GBP", "paris": "EUR", "rome": "EUR",
            "madrid": "EUR", "barcelona": "EUR"}

MAX_SHORT_STAY_NIGHTS = 29  # short-term scope threshold (pending team approval)
PRICE_FLOOR = 10            # data-validity floor in local currency (pending team approval)
FENCE_K = 3                 # log-IQR fence multiplier for price outliers               # log-IQR fence multiplier for price outliers

In [7]:
data_file = next((path for path in DATA_CANDIDATES if path.exists()), None)
if data_file is None:
    raise FileNotFoundError("hostwise_5cities.csv not found - update DATA_CANDIDATES.")

df_all = pd.read_csv(data_file, low_memory=False)

# Drop New York: project covers five cities only
df_all = df_all[df_all["city"].isin(CITY_ORDER)].copy()

# Validation: the file must contain exactly the five project cities (no New York).
assert set(df_all["city"].unique()) == set(CITY_ORDER), sorted(df_all["city"].unique())
print(f"df_all: {df_all.shape[0]:,} listings x {df_all.shape[1]} columns")
print(df_all["city"].value_counts().reindex(CITY_ORDER).to_string())

df_all: 245,402 listings x 91 columns
city
london       92638
paris        77679
rome         37084
madrid       22708
barcelona    15293


In [8]:
# Derived target columns. The original `price` column is never modified.
df_all["price_num"] = pd.to_numeric(
    df_all["price"].astype(str).str.replace(r"[$,]", "", regex=True), errors="coerce"
)
df_all["log_price"] = np.log1p(df_all["price_num"])

# Analysis populations (identical definitions in every HostWise notebook)
df_active = df_all[df_all["source"] == "city scrape"]
df_priced = df_active[df_active["price_num"].notna()]
df_scope = df_priced[df_priced["minimum_nights"] <= MAX_SHORT_STAY_NIGHTS]

print(f"df_active: {len(df_active):,} | df_priced: {len(df_priced):,} | df_scope: {len(df_scope):,}")

df_active: 174,922 | df_priced: 168,725 | df_scope: 153,812


In [9]:
populations = {"df_all": df_all, "df_active": df_active, "df_scope": df_scope}
empty_columns = df_all.columns[df_all.isna().all()].tolist()
print(f"{len(empty_columns)} columns are 100% empty in df_all (listed in Track A, section A5)")

12 columns are 100% empty in df_all (listed in Track A, section A5)


## C1. Missing values by population

**Research question:** Which columns have missing values, and how much of that missingness disappears once we restrict to active listings and to the short-term scope?

**Purpose:** The same column can look unusable in the full data and almost complete in the modelling population. Earlier notebooks reported `bathrooms` as 33.4% missing (full data) and 6.6% missing (active listings) without explaining the difference. Comparing populations side by side shows which missingness is caused by the inactive `previous scrape` rows and which must still be handled on Day 2.

In [10]:
missing_by_population = pd.DataFrame(
    {name: frame.isna().mean() * 100 for name, frame in populations.items()}
)
missing_by_population = (
    missing_by_population
    .drop(index=empty_columns + ["price_num", "log_price"])
    .query("df_all > 0")
    .sort_values("df_all", ascending=False)
)
missing_by_population

,df_all,df_active,df_scope
neighbourhood_group_cleansed,84.51,81.94,84.16
license,49.87,43.20,44.58
host_about,49.17,47.96,48.95
bathrooms,33.42,6.59,6.35
beds,30.33,2.26,2.26
price_quote_total_price,27.67,3.54,0.00
price_quote_price_per_night,27.67,3.54,0.00
estimated_revenue_l365d,27.67,3.54,0.00
price,27.67,3.54,0.00
price_quote_raw,24.77,0.36,0.00


In [ ]:
top_missing = missing_by_population.head(15).iloc[::-1]

ax = top_missing.plot(kind="barh", figsize=(9, 7), width=0.8)
ax.set(title="Missing share by population (15 most-missing non-empty columns)",
       xlabel="% missing", ylabel="")
ax.legend(title="Population")
plt.tight_layout()
plt.savefig(FIG_DIR / "c1_missing_by_population.png", dpi=150)
plt.show()

### Findings

- **Full data (`df_all`):** after the 12 empty columns, the most-missing columns are `neighbourhood_group_cleansed` 84.51%, `license` 49.87%, `host_about` 49.17%, `bathrooms` 33.42%, `beds` 30.33%, `price` 27.67% (also `price_quote_price_per_night`, `price_quote_total_price` and `estimated_revenue_l365d`), `host_location` 24.10%, `bedrooms` 21.59%, and the review columns ~20.1%.
- **Active listings (`df_active`):** missingness collapses for the size columns - `bathrooms` 33.4% -> 6.6%, `beds` 30.3% -> 2.3%, `bedrooms` 21.6% -> 18.0%. Review scores fall slightly to 18.6% (from ~20.1%) and `license` falls to 43.2%.
- The size-column gap is explained in C2: `previous scrape` rows have no `beds` or `bathrooms` at all.

> **Pending execution:** record the `df_scope` column of the table (short-term scope). It is the population the Day 2 imputation will actually face.

### Conclusion

- Missing-value statistics must always be quoted together with their population. For Day 2, only the `df_scope` column is relevant.
- `neighbourhood_group_cleansed` (>80% missing) is unusable. `host_about` and `host_location` are free text and out of scope for v1.
- `bedrooms` (18.0% active) is the only core size feature with substantial missingness after scoping; its pattern is analysed in C8.

## C2. Target missingness and data source

**Research question:** Is a missing price random, or is it concentrated in a specific type of listing?

**Purpose:** If missing prices are random, they are simply dropped. If they belong to a different population (for example, listings no longer active), then both the missing target and the other gaps in those rows are a scope question, not an imputation question. The target is never imputed.

In [ ]:
price_missing_by_source = (
    df_all.groupby("source")["price_num"]
    .agg(listings="size", missing_price=lambda prices: prices.isna().sum())
)
price_missing_by_source["missing_%"] = (
    price_missing_by_source["missing_price"] / price_missing_by_source["listings"] * 100
)
price_missing_by_source

In [ ]:
source_mix = (
    pd.crosstab(df_all["city"], df_all["source"], normalize="index")
    .mul(100).reindex(CITY_ORDER)
)
source_mix

In [ ]:
price_missing_by_city = pd.DataFrame({
    "df_all_%": df_all["price_num"].isna().groupby(df_all["city"]).mean() * 100,
    "df_active_%": df_active["price_num"].isna().groupby(df_active["city"]).mean() * 100,
}).reindex(CITY_ORDER)
price_missing_by_city

In [ ]:
ax = price_missing_by_city.plot(kind="bar", figsize=(8, 4), rot=0)
ax.set(title="Share of listings without a price, by city and population",
       xlabel="", ylabel="% without price")
ax.legend(["All listings (df_all)", "Active listings (df_active)"])
plt.tight_layout()
plt.savefig(FIG_DIR / "c2_missing_price_by_city.png", dpi=150)
plt.show()

In [ ]:
SOURCE_CHECK_COLUMNS = ["price_num", "bathrooms", "beds", "bedrooms", "review_scores_rating"]

missing_by_source = df_all[SOURCE_CHECK_COLUMNS].isna().groupby(df_all["source"]).mean().mul(100)
missing_by_source

In [ ]:
PROFILE_COLUMNS = ["accommodates", "bedrooms", "number_of_reviews", "availability_365"]

has_price_all = df_all["price_num"].notna().rename("has_price")
df_all.groupby(has_price_all)[PROFILE_COLUMNS].mean()

In [ ]:
has_price_active = df_active["price_num"].notna().rename("has_price")
df_active.groupby(has_price_active)[PROFILE_COLUMNS].mean()

### Findings

- **Source explains the missing target.** `previous scrape` rows (70,480 listings) have **87.6%** missing price (61,712 rows); `city scrape` rows (174,922) have only **3.5%** (6,197 rows). 90.9% of all missing prices come from `previous scrape` rows.
- **Per city (all listings):** Paris 37.7%, London 32.8%, Madrid 15.6%, Barcelona 12.7%, Rome 7.4%. **Among active listings:** Paris 6.1%, London 3.6%, Barcelona 2.1%, Madrid 1.5%, Rome 1.4%.
- **`previous scrape` rows have no `beds` and no `bathrooms` at all** (100% missing, 70,480 of 70,480), and `bedrooms` is missing in about 30.6% of them. This explains the C1 gap between populations.
- **Listings without a price look inactive (all listings):** mean `availability_365` 32.1 vs 214.7 days and mean `number_of_reviews` 17.1 vs 47.3, while size is similar (`accommodates` 3.0 vs 3.6; `bedrooms` 1.6 vs 1.7). Strong inference: most of them are delisted or paused listings, not random gaps.

> **Pending execution:** record the `source_mix` table (share of `previous scrape` per city) and the active-only has-price comparison. The active comparison shows whether the remaining 3.5% of missing prices also look inactive.

### Conclusion

- **Evidence-based:** restrict modelling to `source == "city scrape"`. `previous scrape` rows are listings outside the current market; they lack the target and two size features.
- Drop the remaining rows without a price; never impute the target.
- Do not describe a missing price as proof that a listing is inactive - it is a strong inference supported by availability and review activity, not a confirmed status.

## C3. Currency per city

**Research question:** In which currency is each city's price quoted?

**Purpose:** The `price` text shows a `$` sign in every city, but that symbol is formatting, not the currency. If the currencies differ, pooled price statistics and cross-city price comparisons are invalid, and the model must learn city-specific price levels.

In [ ]:
quote_currency = df_active["price_quote_raw"].astype(str).str.extract(r"(GBP|EUR|USD|£|€)")[0]

pd.crosstab(df_active["city"], quote_currency.fillna("not found")).reindex(CITY_ORDER)

In [ ]:
quote_examples = (
    df_active.groupby("city")["price_quote_raw"]
    .apply(lambda quotes: quotes.dropna().head(1).tolist())
    .reindex(CITY_ORDER)
)
quote_examples

### Findings

- **Confirmed (earlier review of `price_quote_raw`):** London prices are in **GBP**; Paris, Rome, Madrid and Barcelona prices are in **EUR**.
- The `$` symbol in `price` appears in every city regardless of currency.

> **Pending execution:** confirm that the crosstab above shows one currency per city and no "not found" rows other than listings without a quote.

### Conclusion

- No pooled price statistic (mean, skew, percentiles) is meaningful across cities. Every price statistic in this project is computed per city.
- The model keeps `city` as a feature, and errors are reported per city. Converting GBP to EUR is unnecessary for modelling because each city has a single currency.

## C4. Target leakage

**Research question:** Which columns are copies of the price, or are calculated from it?

**Purpose:** A column derived from the target makes the model look accurate in testing while being useless in production, where a new host does not yet have a price. Leakage must be verified with data, not assumed from column names.

In [ ]:
quote_per_night = pd.to_numeric(df_all["price_quote_price_per_night"], errors="coerce")
has_both = df_all["price_num"].notna() & quote_per_night.notna()

print(f"Rows with both price and quote: {has_both.sum():,}")
print(f"Share where the quote equals the price: {np.isclose(df_all.loc[has_both, 'price_num'], quote_per_night[has_both]).mean():.4f}")

In [ ]:
pd.crosstab(
    df_all["price_num"].isna().rename("price_missing"),
    df_all["estimated_revenue_l365d"].isna().rename("revenue_missing"),
)

In [ ]:
has_occupancy = df_active["estimated_occupancy_l365d"] > 0
implied_nightly_revenue = df_active["estimated_revenue_l365d"] / df_active["estimated_occupancy_l365d"]
revenue_to_price_ratio = (implied_nightly_revenue / df_active["price_num"])[has_occupancy]

revenue_to_price_ratio.describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99])

### Findings

- **`price_quote_price_per_night` is an exact copy of the target:** it equals `price` on **100%** of the 177,492 rows that have both.
- **`estimated_revenue_l365d` is missing exactly when the price is missing:** both have 67,909 missing values (27.67%) in the full data.

> **Pending execution:** record the crosstab (expected: no rows where only one of the two is missing) and the median of `revenue_to_price_ratio`. A median close to 1.0 confirms that revenue = price x estimated occupancy, i.e. revenue is computed from the target.

### Conclusion

- **Evidence-based:** exclude all `price_quote_*` columns and `estimated_revenue_l365d` from the features.
- **Pending team decision:** `availability_*` and `estimated_occupancy_l365d` are not calculated from price, but they describe demand **at the current price**. A new host cannot know them before setting a price, so they are excluded from v1.

## C5. Duplicates and host concentration

**Research question:** Does the same property appear more than once, and how much of the supply comes from large multi-listing hosts?

**Purpose:** Near-identical listings that land in both training and test data inflate the test score. If many listings belong to the same host, a random split leaks host-specific pricing, and a split grouped by `host_id` becomes necessary.

In [ ]:
print(f"Exact duplicate rows (df_all): {df_all.duplicated().sum():,}")
print(f"Duplicate listing IDs (df_all): {df_all['id'].duplicated().sum():,}")

In [ ]:
identical_except_ids = (
    df_active.drop(columns=["id", "listing_url", "picture_url"])
    .astype(str).duplicated().sum()
)
print(f"Active rows identical to another row except id/URLs: {identical_except_ids:,}")

In [ ]:
TEMPLATE_KEY = ["host_id", "name", "room_type", "accommodates", "bedrooms", "bathrooms_text", "price_num"]

is_template_duplicate = df_active.duplicated(TEMPLATE_KEY, keep=False)
print(f"Active rows sharing a host + listing template: {is_template_duplicate.sum():,} ({is_template_duplicate.mean():.1%})")

In [ ]:
is_template_duplicate.groupby(df_active["city"]).mean().mul(100).reindex(CITY_ORDER).rename("template_duplicate_%").to_frame()

In [ ]:
host_portfolio_size = df_active.groupby("host_id")["id"].transform("size")
host_size_band = pd.cut(host_portfolio_size, [0, 1, 5, 50, np.inf], labels=["1", "2-5", "6-50", "50+"])

host_concentration = (
    pd.crosstab(df_active["city"], host_size_band, normalize="index")
    .mul(100).reindex(CITY_ORDER)
)
host_concentration

In [ ]:
ax = host_concentration.sort_values("1").plot(
    kind="barh", stacked=True, figsize=(9, 4), colormap="viridis"
)
ax.set(title="Who supplies listings? Share of active listings by host portfolio size",
       xlabel="% of active listings", ylabel="")
ax.legend(title="Host's listings", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.savefig(FIG_DIR / "c5_host_concentration.png", dpi=150)
plt.show()

### Findings

- **No exact duplicates:** 0 duplicate rows and 0 duplicate listing IDs in the full data.
- **33 active rows** are identical to another row in everything except the ID and URLs.
- **1,227 active rows (0.7%)** share a host and an identical listing template (name, room type, capacity, bedrooms, bath text and price): Madrid 2.1%, Barcelona 1.5%, London 0.6%, Paris 0.4%, Rome 0.3%. Strong inference: most are multi-unit properties (several identical flats in one building), not data errors.
- **Host concentration differs strongly by city.** Hosts with 6+ listings supply 64.2% of active listings in Barcelona, 53.3% in Madrid, 37.7% in London, 28.2% in Paris and 27.3% in Rome. Single-listing hosts are 57.5% of Paris but only 15.8% of Barcelona.

### Conclusion

- Do not delete the template duplicates: they are plausibly real, separately bookable units.
- **Proposed (pending team approval):** split train/test **grouped by `host_id`**. With up to 24.6% of a city's listings from 50+ listing hosts, a random split would let the model memorise a host's pricing template and overstate accuracy.

## C6. Invalid, implausible and inconsistent values

**Research question:** Which values are impossible, implausible, or contradict another column?

**Purpose:** Impossible values are errors; implausible values need verification; cross-column contradictions reveal data-entry problems that single-column checks miss. Counting them per city decides whether Day 2 should drop, cap or flag them.

In [ ]:
impossible_checks = {
    "price <= 0": df_all["price_num"] <= 0,
    "accommodates <= 0": df_all["accommodates"] <= 0,
    "bedrooms < 0": df_all["bedrooms"] < 0,
    "bathrooms < 0": df_all["bathrooms"] < 0,
    "minimum_nights > 365": df_all["minimum_nights"] > 365,
    "availability_365 outside 0-365": ~df_all["availability_365"].between(0, 365) & df_all["availability_365"].notna(),
    "latitude outside -90..90": ~df_all["latitude"].between(-90, 90),
    "longitude outside -180..180": ~df_all["longitude"].between(-180, 180),
}
pd.Series({check: int(mask.sum()) for check, mask in impossible_checks.items()}, name="rows (df_all)").to_frame()

In [ ]:
df_all.loc[df_all["minimum_nights"] > 365, "source"].value_counts()

In [ ]:
price = df_active["price_num"]

implausible_checks = pd.DataFrame({
    "price < 10": price < 10,
    "price > 5000": price > 5000,
    "bedrooms > 10": df_active["bedrooms"] > 10,
    "beds > 20": df_active["beds"] > 20,
    "bathrooms > 10": df_active["bathrooms"] > 10,
    "review_score = 0": df_active["review_scores_rating"] == 0,
    "reviews_per_month > 30": df_active["reviews_per_month"] > 30,
    "maximum_nights > 1125": df_active["maximum_nights"] > 1125,
})
implausible_checks.groupby(df_active["city"]).sum().reindex(CITY_ORDER)

In [ ]:
is_entire_home = df_active["room_type"] == "Entire home/apt"
bath_text = df_active["bathrooms_text"]

inconsistency_checks = pd.DataFrame({
    "bedrooms > accommodates": df_active["bedrooms"] > df_active["accommodates"],
    "beds > 3 x accommodates": df_active["beds"] > 3 * df_active["accommodates"],
    "entire home with shared bath": is_entire_home & bath_text.str.contains("shared", case=False, na=False),
    "entire home with 0 baths": is_entire_home & bath_text.str.startswith("0", na=False),
})
inconsistency_checks.groupby(df_active["city"]).sum().reindex(CITY_ORDER)

In [ ]:
zero_bath_entire = inconsistency_checks["entire home with 0 baths"]
df_active.loc[zero_bath_entire, "bathrooms"].value_counts(dropna=False).head(10).rename("listings").to_frame()

In [ ]:
pd.crosstab(
    df_active["bathrooms"].isna().rename("bathrooms_missing"),
    df_active["bathrooms_text"].isna().rename("bathrooms_text_missing"),
)

### Findings

**Impossible values (full data)**
- None for price, capacity, bedrooms, bathrooms, availability or coordinates.
- **52 rows have `minimum_nights > 365`.** All of them are `previous scrape` rows: the maximum `minimum_nights` among active listings is 95 (Track A, A2). They are therefore already outside the modelling population.

**Implausible values (active listings, per city)**

| Check | Barcelona | London | Madrid | Paris | Rome |
|---|---|---|---|---|---|
| price < 10 | 0 | 1 | 4 | 2 | 2 |
| price > 5000 | 6 | 100 | 28 | 110 | 39 |
| bedrooms > 10 | 59 | 16 | 7 | 9 | 12 |
| beds > 20 | 26 | 9 | 13 | 4 | 5 |
| bathrooms > 10 | 12 | 14 | 2 | 5 | 6 |
| review_score = 0 | 0 | 0 | 0 | 1 | 0 |
| reviews_per_month > 30 | 5 | 0 | 5 | 3 | 1 |
| maximum_nights > 1125 | 0 | 16 | 1 | 3 | 5 |

**Inconsistencies (active listings, per city)**

| Check | Barcelona | London | Madrid | Paris | Rome |
|---|---|---|---|---|---|
| bedrooms > accommodates | 117 | 416 | 151 | 69 | 111 |
| beds > 3 x accommodates | 47 | 77 | 56 | 12 | 56 |
| entire home with shared bath | 0 | 0 | 0 | 0 | 0 |
| entire home with 0 baths | 460 | 303 | 254 | 272 | 50 |

- Barcelona has the most implausible size values relative to its size (59 listings with >10 bedrooms out of ~15k).
- 1,339 entire homes report "0 baths". Barcelona alone has 460.

> **Pending execution:** record (1) the numeric `bathrooms` value of the "0 baths" entire homes and (2) the `bathrooms` vs `bathrooms_text` missing crosstab. They show whether "0 baths" means "really zero" or "not filled in", and whether `bathrooms_text` can always replace `bathrooms`.

### Conclusion

- Implausible size values are **candidates for verification, not automatic errors**. Proposed Day 2 rule (pending): cap `bedrooms`, `beds` and bath counts at a per-feature limit and add a flag, rather than dropping rows.
- "bedrooms > accommodates" (864 rows) can be legitimate (for example, a large house rented for few guests) and is kept.
- `maximum_nights` contains placeholders and has little pricing meaning; it is a candidate for exclusion.
- The 52 `minimum_nights > 365` rows need no action because the `city scrape` filter already removes them.

## C7. Stay length: short-term vs mid/long-term listings

**Research question:** What share of each city's active listings require 30 or more nights, and are they priced differently per night?

**Purpose:** A 30+ night rental is a different product (monthly lease) with a lower nightly rate. Mixing both products makes the model learn two pricing logics at once. This evidence supports - or rejects - the proposed short-term scope (`minimum_nights <= 29`).

In [ ]:
stay_length = pd.cut(df_active["minimum_nights"], [0, 7, 29, np.inf], labels=["<=7", "8-29", "30+"])

stay_mix = pd.crosstab(df_active["city"], stay_length, normalize="index").mul(100).reindex(CITY_ORDER)
stay_mix

In [ ]:
median_price_by_stay = (
    df_active.groupby(["city", stay_length], observed=True)["price_num"]
    .median().unstack().reindex(CITY_ORDER)
)
median_price_by_stay

In [ ]:
long_stay_discount = (1 - median_price_by_stay["30+"] / median_price_by_stay["<=7"]).mul(100).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
stay_mix.sort_values("30+").plot(kind="barh", stacked=True, ax=axes[0], colormap="viridis")
axes[0].set(title="Share of active listings by minimum stay", xlabel="% of active listings", ylabel="")
axes[0].legend(title="Min. nights", bbox_to_anchor=(1, 1))

long_stay_discount.plot(kind="barh", ax=axes[1], color="#d95f02")
axes[1].set(title="30+ night listings are cheaper per night",
            xlabel="% lower median nightly price vs <=7 nights (same city)", ylabel="")
plt.tight_layout()
plt.savefig(FIG_DIR / "c7_stay_length.png", dpi=150)
plt.show()

### Findings

| City | <=7 nights | 8-29 nights | 30+ nights | Median <=7 | Median 30+ | 30+ discount |
|---|---|---|---|---|---|---|
| Barcelona | 61.0% | 2.4% | **36.7%** | 260 EUR | 79 EUR | -70% |
| Paris | 85.0% | 1.4% | 13.6% | 231 EUR | 88 EUR | -62% |
| Madrid | 82.8% | 6.2% | 10.9% | 134 EUR | 88 EUR | -34% |
| Rome | 94.2% | 0.2% | 5.6% | 166 EUR | 96 EUR | -42% |
| London | 95.2% | 3.8% | 1.0% | 184 GBP | 144 GBP | -22% |

- In every city, 30+ night listings have a lower median nightly price than short stays (comparison within each city and currency).
- Barcelona is the most affected: more than a third of its active listings are mid/long-term, and their median nightly price is 70% lower.
- 14 active listings have no `minimum_nights` value and fall outside the scope filter.

### Conclusion

- The evidence supports the short-term scope `minimum_nights <= 29`: 30+ night listings are a different product with a different price level. Without the filter, Barcelona's model would mix two markets.
- **Pending team decision:** approve the threshold. HostWise would then be presented as a **short-term rental** pricing assistant.
- Correlation, not causation: the lower nightly price may reflect the product (monthly lease), the listing type, or both.

## C8. Structural missingness

**Research question:** Do missing values follow a pattern that carries meaning (structural), or are they random?

**Purpose:** Structural missingness is information. Filling a structurally missing value with a global average erases that signal and biases the feature. The hypotheses tested here are: (1) missing review scores = listings with no reviews; (2) missing `bedrooms` is concentrated in specific room or property types and is related to price.

In [ ]:
pd.crosstab(
    df_active["review_scores_rating"].isna().rename("score_missing"),
    (df_active["number_of_reviews"] == 0).rename("zero_reviews"),
)

In [ ]:
SIZE_COLUMNS = ["bedrooms", "beds", "bathrooms", "bathrooms_text", "price_num"]

df_active[SIZE_COLUMNS].isna().groupby(df_active["city"]).mean().mul(100).reindex(CITY_ORDER)

In [ ]:
df_active["bedrooms"].isna().groupby(df_active["room_type"]).mean().mul(100).rename("bedrooms_missing_%").to_frame()

In [ ]:
df_active.loc[df_active["bedrooms"].isna(), "property_type"].value_counts().head(8).rename("listings").to_frame()

In [ ]:
bedrooms_missing = df_active["bedrooms"].isna().rename("bedrooms_missing")

df_active.groupby(["city", bedrooms_missing])["price_num"].median().unstack().reindex(CITY_ORDER)

In [ ]:
df_active.groupby(["room_type", bedrooms_missing])["price_num"].median().unstack()

### Findings

- **Review scores:** all 32,513 listings with a missing rating have 0 reviews. 7 listings have 0 reviews but a rating (a minor inconsistency). Missing review scores therefore mean **"no reviews yet"** - the correct label is `has_no_reviews`, not "new listing", because listing age is not established.
- **Missing `bedrooms` by room type:** Private room 57.8%, Shared room 30.4%, Hotel room 16.6%, Entire home/apt 6.5%. The most common property types among missing values are "Private room in rental unit" (11,052), "Entire rental unit" (7,658) and "Private room in home" (5,410).
- **Missing `bedrooms` is related to price.** Median price with vs without `bedrooms`: Barcelona 202 vs 47 EUR, London 221 vs 85 GBP, Madrid 140 vs 74 EUR, Paris 229 vs 132 EUR, Rome 176 vs 114 EUR. This is largely a room-type effect: private rooms are both cheaper and more often missing `bedrooms`.
- **Missing by city (active):** `bedrooms` 11.4-21.7%, `beds` 1.4-2.7%, `bathrooms` 5.9-10.0%, `bathrooms_text` 0.1-0.2%.

> **Pending execution:** record the median price by room type x bedrooms-missing table. If the gap shrinks within room type, missingness is mainly a room-type effect; if a gap remains, a `bedrooms_missing` flag adds information.

### Conclusion

- Review scores: add a `has_no_reviews` flag and fill the scores with a neutral value on Day 2 (fitted on the training data only).
- `bedrooms`: impute **within room type** (not with a global median) and add a missing flag. A private-room listing usually means one bedroom for the guest.
- `bathrooms`: replace with counts parsed from `bathrooms_text`, which is almost complete (<= 0.2% missing).
- All imputation values are learned on the training split only.

## C9. Price outliers (investigation only)

**Research question:** How extreme are the price tails in each city, and which outlier rule separates plausible prices from likely errors without removing real luxury listings?

**Purpose:** A few extreme prices (for example, 527,524 GBP for one night) distort means and the training loss. But a fixed global threshold ignores that cities differ in price level and currency. This section compares candidate rules **per city** on the short-term scope. No rows are removed here; the rule is a Day 2 decision.

In [ ]:
df_priced.groupby("city")["price_num"].agg(["count", "median", "mean", "max"]).reindex(CITY_ORDER)

In [ ]:
pd.DataFrame({
    "skew_price": df_priced.groupby("city")["price_num"].skew(),
    "skew_log_price": df_priced.groupby("city")["log_price"].skew(),
}).reindex(CITY_ORDER)

In [ ]:
TAIL_QUANTILES = [0, 0.001, 0.005, 0.01, 0.5, 0.99, 0.995, 0.999, 1]

tail_percentiles = (
    df_scope.groupby("city")["price_num"].quantile(TAIL_QUANTILES)
    .unstack().reindex(CITY_ORDER)
)
tail_percentiles.columns = ["min", "p0.1", "p0.5", "p1", "median", "p99", "p99.5", "p99.9", "max"]
tail_percentiles

In [ ]:
df_scope.nlargest(10, "price_num")[["city", "room_type", "accommodates", "bedrooms", "minimum_nights", "price_num"]]

In [ ]:
def count_outlier_flags(prices: pd.Series) -> pd.Series:
    log_prices = np.log(prices)
    q1, q3 = log_prices.quantile([0.25, 0.75])
    iqr = q3 - q1
    p_low, p_high = prices.quantile([0.005, 0.995])
    return pd.Series({
        "A: fixed <10 or >5000": ((prices < 10) | (prices > 5000)).sum(),
        "B: outside p0.5-p99.5": ((prices < p_low) | (prices > p_high)).sum(),
        "C: log-IQR fence (3x)": ((log_prices < q1 - FENCE_K * iqr) | (log_prices > q3 + FENCE_K * iqr)).sum(),
        "rows": len(prices),
    })

In [ ]:
df_scope.groupby("city")["price_num"].apply(count_outlier_flags).unstack().reindex(CITY_ORDER)

In [ ]:
def log_iqr_fence(prices: pd.Series, k: float = FENCE_K) -> pd.Series:
    log_prices = np.log(prices)
    q1, q3 = log_prices.quantile([0.25, 0.75])
    iqr = q3 - q1
    return pd.Series({"lower": np.exp(q1 - k * iqr), "upper": np.exp(q3 + k * iqr)})


price_fences = df_scope.groupby("city")["price_num"].apply(log_iqr_fence).unstack().reindex(CITY_ORDER)
price_fences["currency"] = price_fences.index.map(CURRENCY)
price_fences

In [ ]:
lower_fence = np.maximum(df_scope["city"].map(price_fences["lower"]), PRICE_FLOOR)
upper_fence = df_scope["city"].map(price_fences["upper"])
inside_fences = df_scope["price_num"].between(lower_fence, upper_fence)

print(f"Inside the log-IQR fences + floor of {PRICE_FLOOR}: {inside_fences.sum():,} of {len(df_scope):,} ({inside_fences.mean():.2%})")
inside_fences.groupby(df_scope["city"]).agg(["sum", "size"]).reindex(CITY_ORDER)

In [ ]:
city_order_by_median = df_scope.groupby("city")["log_price"].median().sort_values().index

fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(data=df_scope, x="log_price", y="city", order=city_order_by_median, fliersize=1, ax=ax)
ax.set(title="Log nightly price per city (local currency) - outliers on both sides",
       xlabel="log(1 + price)", ylabel="")
plt.tight_layout()
plt.savefig(FIG_DIR / "c9_log_price_outliers.png", dpi=150)
plt.show()

### Findings

**Active priced listings (`df_priced`, local currency)**

| City | Listings | Median | Mean | Max |
|---|---|---|---|---|
| London (GBP) | 59,345 | 183.00 | 276.06 | 527,524.00 |
| Paris (EUR) | 45,215 | 210.00 | 330.75 | 97,003.05 |
| Rome (EUR) | 33,136 | 163.67 | 223.51 | 13,728.50 |
| Madrid (EUR) | 18,263 | 126.00 | 182.86 | 36,883.00 |
| Barcelona (EUR) | 12,766 | 181.34 | 244.90 | 10,542.22 |

- In every city the mean is far above the median: price is strongly right-skewed. This supports modelling `log_price`.

**Short-term scope (`df_scope`, 153,812 listings)**

| City | p1 | Median | p99 | p99.9 | Max | Rule A | Rule B | Rule C | Lower fence | Upper fence |
|---|---|---|---|---|---|---|---|---|---|---|
| London | 38 | 183 | 1,370 | 6,300 | 527,524 | 99 | 589 | 36 | 4 | 8,399 |
| Paris | 67 | 230 | 2,067 | 8,021 | 97,003 | 105 | 394 | 102 | 11 | 5,282 |
| Rome | 62 | 166 | 1,082 | 4,799 | 13,728 | 28 | 311 | 149 | 17 | 1,732 |
| Madrid | 25 | 131 | 999 | 9,130 | 36,883 | 30 | 164 | 63 | 9 | 1,872 |
| Barcelona | 46 | 255 | 2,283 | 4,070 | 7,532 | 5 | 80 | 82 | 27 | 2,395 |

- **Rule A (fixed thresholds)** ignores city price levels: it flags 105 Paris listings but only 5 in Barcelona.
- **Rule B (percentiles)** always removes 1% of each city by construction, including real high-end listings (589 in London).
- **Rule C (log-IQR fence, k = 3)** adapts to each city's spread and flags only extreme values: 432 rows in total. In the original notebook, 153,382 rows (99.72%) were inside the rounded fences before any price floor.
- **London's lower fence (4 GBP) is not a plausible nightly price**, so a floor of 10 (local currency) is added to the rule.

> **Pending execution:** record the per-city skew table, the 10 most expensive scope listings, and the exact count inside "fences + floor" (expected about 153,380; the fences here are not rounded).

### Conclusion

- **Proposed rule (pending team approval):** per-city log-IQR fence with k = 3, plus a floor of 10 in local currency, applied to the **training data only**.
- The fences are recomputed on the training split on Day 2; the values above are for exploration only.
- **Test-set policy:** test listings are never removed because their true price falls outside fences learned on training data. If a validity rule is needed for the test set (for example, the price floor), it must be defined in advance and applied identically, without looking at model errors.
- Track B uses the same rule to build its EDA population `df_eda`, so the charts are not dominated by a handful of extreme prices.

## Track C summary

| Issue | Evidence | Day 2 action | Status |
|---|---|---|---|
| Missing target | 87.6% missing in `previous scrape` vs 3.5% in `city scrape` | Keep `city scrape`; drop missing prices; never impute the target | Evidence-based |
| Currency | London GBP, other cities EUR | Per-city statistics and per-city error reporting | Evidence-based |
| Target leakage | Quote = price on 100% of 177,492 rows; revenue missing on the same rows | Exclude `price_quote_*`, `estimated_revenue_l365d` | Evidence-based |
| Demand columns | Describe demand at the current price | Exclude `availability_*`, `estimated_occupancy_l365d` in v1 | Pending team approval |
| Duplicates | 0 exact; 1,227 template rows (0.7%), plausibly multi-unit | Keep | Evidence-based |
| Host concentration | 6+ listing hosts supply 27-64% of a city's listings | Split grouped by `host_id` | Pending team approval |
| Implausible sizes | e.g. 59 Barcelona listings with >10 bedrooms | Cap + flag | Pending team approval |
| Stay length | 30+ nights: 36.7% of Barcelona, 22-70% cheaper per night | Scope `minimum_nights <= 29` | Pending team approval |
| Review scores | Missing exactly when there are no reviews | `has_no_reviews` flag + neutral fill | Evidence-based |
| `bedrooms` | 18.0% missing, concentrated in private rooms | Impute within room type + flag | Evidence-based |
| `bathrooms` | 6.6% missing; text almost complete | Parse `bathrooms_text` | Evidence-based |
| Price outliers | Max 527,524 GBP; fixed thresholds ignore city levels | Per-city log-IQR fence (k = 3) + floor 10, training data only | Pending team approval |

**Next:** Track B explores which features relate to price, using the short-term scope with the outlier rule applied for exploration only.